Bronze Layer — Raw volume to Bronze
Loads every .json.gz file in the raw volume into bronze_nyc311_new. Each file becomes its own batch, with the batch_id taken from the filename, so page_3.json.gz becomes batch_id = page_3.

## 1. Schema (explicit, no inference)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
  StructField("unique_key", StringType(), False),
  StructField("created_date", TimestampType()),
  StructField("closed_date", TimestampType()),
  StructField("name", StringType()),
  StructField("agency", StringType()),
  StructField("agency_name", StringType()),
  StructField("complaint_type", StringType()),
  StructField("descriptor", StringType()),
  StructField("descriptor_2", StringType()),
  StructField("location_type", StringType()),
  StructField("incident_zip", StringType()),
  StructField("incident_address", StringType()),
  StructField("street_name", StringType()),
  StructField("cross_street_1", StringType()),
  StructField("cross_street_2", StringType()),
  StructField("intersection_street_1", StringType()),
  StructField("intersection_street_2", StringType()),
  StructField("address_type", StringType()),
  StructField("city", StringType()),
  StructField("landmark", StringType()),
  StructField("facility_type", StringType()),
  StructField("status", StringType()),
  StructField("due_date", TimestampType()),
  StructField("resolution_description", StringType()),
  StructField("resolution_action_updated_date", TimestampType()),
  StructField("community_board", StringType()),
  StructField("council_district", StringType()),
  StructField("fbicr_code", StringType()),
  StructField("police_precinct", StringType()),
  StructField("latitude", StringType()),
  StructField("bbl", StringType()),
  StructField("borough", StringType()),
  StructField("location", StringType()),
  StructField("x_coordinate", StringType()),
  StructField("y_coordinate", StringType()),
  StructField("longitude", StringType()),
  StructField("open_data_channel_type", StringType()),
  StructField("park_facility_name", StringType()),
  StructField("park_borough", StringType()),
  StructField("vehicle_type", StringType()),
  StructField("taxi_company_borough", StringType()),
  StructField("taxi_pick_up_location", StringType()),
  StructField("bridge_highway_name", StringType()),
  StructField("bridge_highway_direction", StringType()),
  StructField("road_ramp", StringType()),
  StructField("bridge_highway_segment", StringType())
])

print(len(schema.fields), "declared fields")

46 declared fields


Data Dictionary


In [0]:
from pyspark.sql.functions import lit

COLUMN_DOCS = {
  "unique_key": "Unique identifier of the service request (primary key)",
  "created_date": "Date and time the service request was created",
  "closed_date": "Date and time the request was closed; null while still open",
  "name": "Reserved source field, not populated in this dataset",
  "agency": "Acronym of the responding city agency",
  "agency_name": "Full name of the responding city agency",
  "complaint_type": "Primary category of the complaint (renamed 'Problem' by NYC in 2025)",
  "descriptor": "Detail describing the complaint type (renamed 'Problem Detail')",
  "descriptor_2": "Additional detail on the complaint",
  "location_type": "Type of location, e.g. street, residential building",
  "incident_zip": "ZIP code of the incident",
  "incident_address": "Street address of the incident (PII, dropped in Silver)",
  "street_name": "Street name of the incident (PII, dropped in Silver)",
  "cross_street_1": "First cross street (PII, dropped in Silver)",
  "cross_street_2": "Second cross street (PII, dropped in Silver)",
  "intersection_street_1": "First intersecting street (PII, dropped in Silver)",
  "intersection_street_2": "Second intersecting street (PII, dropped in Silver)",
  "address_type": "How the location was specified: address, intersection, block face",
  "city": "City of the incident",
  "landmark": "Nearest landmark to the incident",
  "facility_type": "Type of facility the request relates to",
  "status": "Current status: Assigned, Pending, Closed, Canceled",
  "due_date": "Date the agency is expected to resolve the request by",
  "resolution_description": "Free text describing how the request was resolved",
  "resolution_action_updated_date": "When the resolution was last updated; recency key",
  "community_board": "Community board responsible for the area",
  "council_district": "City council district number",
  "fbicr_code": "Reserved source field, not populated in this dataset",
  "police_precinct": "NYPD precinct covering the incident location",
  "latitude": "Latitude of the incident (PII, dropped in Silver)",
  "bbl": "Borough-block-lot parcel number (PII, dropped in Silver)",
  "borough": "Borough of the incident",
  "location": "Combined coordinate object, flattened to text at ingestion",
  "x_coordinate": "State plane X coordinate (PII, dropped in Silver)",
  "y_coordinate": "State plane Y coordinate (PII, dropped in Silver)",
  "longitude": "Longitude of the incident (PII, dropped in Silver)",
  "open_data_channel_type": "How the request was reported: phone, online, mobile, other",
  "park_facility_name": "Name of the park facility, where applicable",
  "park_borough": "Borough of the park facility",
  "vehicle_type": "Vehicle type for taxi-related complaints",
  "taxi_company_borough": "Borough of the taxi company",
  "taxi_pick_up_location": "Taxi pick-up location",
  "bridge_highway_name": "Bridge or highway name, where applicable",
  "bridge_highway_direction": "Direction of travel on the bridge or highway",
  "road_ramp": "Whether the location is a road or a ramp",
  "bridge_highway_segment": "Segment of the bridge or highway",
  "source_file": "Full path of the raw file this record was read from (lineage)",
  "batch_id": "Batch this record belongs to, taken from the filename (partition key)",
  "load_timestamp": "When this record was ingested into Bronze",
}

PRIMARY_KEY = ["unique_key", "batch_id"]

BRONZE = "workspace.default.bronze_nyc311_new"  # defined here because this section runs before the Parameters section

bronze_cols = spark.table(BRONZE).schema.fields

dictionary = spark.createDataFrame(
    [(f.name,
      f.dataType.simpleString(),
      "YES" if f.nullable else "NO",
      "PK" if f.name in PRIMARY_KEY else "",
      COLUMN_DOCS.get(f.name, "Added by schema drift handling"))
     for f in bronze_cols],
    ["column_name", "data_type", "nullable", "key", "description"]
)

dictionary.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.bronze_data_dictionary")

display(dictionary)

column_name,data_type,nullable,key,description
unique_key,string,YES,PK,Unique identifier of the service request (primary key)
created_date,timestamp,YES,,Date and time the service request was created
closed_date,timestamp,YES,,Date and time the request was closed; null while still open
name,string,YES,,"Reserved source field, not populated in this dataset"
agency,string,YES,,Acronym of the responding city agency
agency_name,string,YES,,Full name of the responding city agency
complaint_type,string,YES,,Primary category of the complaint (renamed 'Problem' by NYC in 2025)
descriptor,string,YES,,Detail describing the complaint type (renamed 'Problem Detail')
descriptor_2,string,YES,,Additional detail on the complaint
location_type,string,YES,,"Type of location, e.g. street, residential building"


## 2. Parameters

Nothing is hardcoded. `raw_path` can be a folder (load everything) or a single
`.json.gz` file. `batch_id` is normally blank and only used to target one file.

In [0]:
import os
from datetime import datetime

dbutils.widgets.text("raw_path", "/Volumes/workspace/default/nyc311_raw_new/")
dbutils.widgets.text("batch_id", "")

raw_path  = dbutils.widgets.get("raw_path").strip()
one_batch = dbutils.widgets.get("batch_id").strip()

BRONZE     = "workspace.default.bronze_nyc311_new"
QUARANTINE = "workspace.default.nyc311_quarantine_new"
LOG_TABLE  = "workspace.default.pipeline_execution_logs_new"

if raw_path.endswith(".json.gz"):
    files = [raw_path]
else:
    base  = raw_path.rstrip("/")
    files = [f"{base}/{f.name}" for f in dbutils.fs.ls(base)
             if f.name.endswith(".json.gz")]

if one_batch:
    files = [f for f in files
             if os.path.basename(f).replace(".json.gz", "") == one_batch]
    if not files:
        raise ValueError(f"{one_batch}.json.gz not found in {raw_path}")

files = sorted(files)

if not files:
    raise ValueError(f"No .json.gz files found in {raw_path}")

print(len(files), "file(s) to load:")
for f in files:
    print("   ", os.path.basename(f))

2 file(s) to load:
    page_3.json.gz
    page_4.json.gz


## 3. Execution log

One row per file processed, recording the layer, the file, start and end time,
status, and how many rows were written. Failures are logged too, so a batch that
crashes still leaves a trace.

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    layer STRING,
    parameter STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    rows_read BIGINT,
    rows_inserted BIGINT,
    rows_quarantined BIGINT,
    error_message STRING
) USING DELTA""")

def write_log(parameter, start, status, rows_read=None,
              rows_inserted=None, rows_quarantined=None, error=None):
    spark.sql(f"""INSERT INTO {LOG_TABLE} VALUES (
        'Raw-to-Bronze',
        '{parameter}',
        TIMESTAMP '{start}',
        current_timestamp(),
        '{status}',
        {rows_read        if rows_read        is not None else 'NULL'},
        {rows_inserted    if rows_inserted    is not None else 'NULL'},
        {rows_quarantined if rows_quarantined is not None else 'NULL'},
        {repr(str(error)[:500]) if error else 'NULL'})""")
    print(f"    [log] {parameter} | {status}")

print("log table ready:", LOG_TABLE)

log table ready: workspace.default.pipeline_execution_logs_new


## 4. Load one file

Everything that happens to a single batch: read with the explicit schema, check
for schema drift, quarantine unparseable rows, write the batch.

`load_timestamp` is stamped on every record here, recording when it was ingested
into Bronze.

In [0]:
from pyspark.sql.functions import (col, lit, current_timestamp,
                                   explode, map_keys, from_json)

def load_one_file(file_path):
    batch_id = os.path.basename(file_path).replace(".json.gz", "")
    start    = datetime.now()
    print(f"\n--- {batch_id} ---")
    tmp_table = f"workspace.default._tmp_staged_{batch_id.replace('-', '_').replace('.', '_')}"

    try:
        schema_with_corrupt = StructType(
            schema.fields + [StructField("_corrupt_record", StringType())])

        raw_df = (spark.read.schema(schema_with_corrupt)
                  .option("mode", "PERMISSIVE")
                  .option("columnNameOfCorruptRecord", "_corrupt_record")
                  .json(file_path))

        # --- schema drift: columns present in the file but not in the contract
        incoming = {r["k"] for r in spark.read.text(file_path)
                    .select(explode(map_keys(
                        from_json(col("value"), "map<string,string>"))).alias("k"))
                    .distinct().collect()}
        unexpected = incoming - {f.name for f in schema.fields}

        if unexpected:
            print("    schema drift - new columns added as string:",
                  sorted(unexpected))
            drift_schema = StructType(
                schema.fields
                + [StructField(c, StringType()) for c in sorted(unexpected)]
                + [StructField("_corrupt_record", StringType())])
            raw_df = (spark.read.schema(drift_schema)
                      .option("mode", "PERMISSIVE")
                      .option("columnNameOfCorruptRecord", "_corrupt_record")
                      .json(file_path))
        else:
            print("    no schema drift")

        staged = (raw_df
                  .withColumn("source_file", col("_metadata.file_path"))
                  .withColumn("batch_id", lit(batch_id))
                  .withColumn("load_timestamp", current_timestamp()))

        # --- materialize via temp table (.cache / PERSIST TABLE not supported on serverless)
        staged.write.format("delta").mode("overwrite").saveAsTable(tmp_table)
        staged = spark.table(tmp_table)

        # --- quarantine anything unparseable instead of failing the batch
        bad = staged.filter(col("_corrupt_record").isNotNull())
        bad_count = bad.count()

        if bad_count > 0:
            print("    quarantining", bad_count, "records")
            quarantine_df = (bad.select(
                                "unique_key",
                                col("_corrupt_record").alias("raw_record"),
                                "source_file", "batch_id", "load_timestamp")
                             .withColumn("reason", lit("unparseable record")))

            quarantine_df.limit(0).write.format("delta") \
                .partitionBy("batch_id").mode("ignore").saveAsTable(QUARANTINE)

            (quarantine_df.write.format("delta")
                .mode("overwrite")
                .option("replaceWhere", f"batch_id = '{batch_id}'")
                .partitionBy("batch_id")
                .saveAsTable(QUARANTINE))

        # --- the good rows
               # --- the good rows
        good = (staged.filter(col("_corrupt_record").isNull())
                      .drop("_corrupt_record")
                      .filter(col("unique_key").isNotNull()))

        rows = good.count()

        good.limit(0).write.format("delta") \
            .partitionBy("batch_id").mode("ignore").saveAsTable(BRONZE)

        (good.write.format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"batch_id = '{batch_id}'")
            .partitionBy("batch_id")
            .saveAsTable(BRONZE))

        print("    loaded", rows, "rows")
        write_log(batch_id, start, "SUCCESS", rows, rows, bad_count)
        spark.sql(f"DROP TABLE IF EXISTS {tmp_table}")
        return rows

    except Exception as e:
        spark.sql(f"DROP TABLE IF EXISTS {tmp_table}")
        write_log(batch_id, start, "FAILURE", error=e)
        print("    FAILED:", str(e)[:200])
        raise

## 5. Run

Each file is loaded independently, so if one fails the batches already loaded
stay intact and you can re-run just the failed one by putting its name in the
`batch_id` widget.

In [0]:
total = 0
for file_path in files:
    total += load_one_file(file_path)

print("\n==============================")
print("Files processed:", len(files))
print("Total rows loaded:", total)


--- page_3 ---
    no schema drift
    loaded 50000 rows
    [log] page_3 | SUCCESS

--- page_4 ---
    no schema drift
    loaded 50000 rows
    [log] page_4 | SUCCESS

Files processed: 2
Total rows loaded: 100000


## 6. Verify

In [0]:
display(spark.table(BRONZE).groupBy("batch_id").count().orderBy("batch_id"))

batch_id,count
page_3,50000
page_4,50000


In [0]:
print("total bronze rows:", spark.table(BRONZE).count())
print("columns:", len(spark.table(BRONZE).columns))

spark.table(BRONZE).select(
    "unique_key", "created_date", "complaint_type", "borough",
    "batch_id", "load_timestamp", "source_file").show(5, truncate=False)

total bronze rows: 100000
columns: 49
+----------+-------------------+---------------+---------+--------+--------------------------+-------------------------------------------------------------+
|unique_key|created_date       |complaint_type |borough  |batch_id|load_timestamp            |source_file                                                  |
+----------+-------------------+---------------+---------+--------+--------------------------+-------------------------------------------------------------+
|70453669  |2026-09-17 07:45:35|Illegal Parking|BROOKLYN |page_4  |2026-10-08 14:43:33.103295|dbfs:/Volumes/workspace/default/nyc311_raw_new/page_4.json.gz|
|70455752  |2026-09-17 07:45:24|Illegal Parking|QUEENS   |page_4  |2026-10-08 14:43:33.103295|dbfs:/Volumes/workspace/default/nyc311_raw_new/page_4.json.gz|
|70446097  |2026-09-17 07:45:18|Illegal Parking|BROOKLYN |page_4  |2026-10-08 14:43:33.103295|dbfs:/Volumes/workspace/default/nyc311_raw_new/page_4.json.gz|
|70446923  |2026-09-

Duplicate `unique_key` values across batches are expected — Bronze keeps every
version that arrived. Silver resolves them down to one row per request.

In [0]:
dups = spark.table(BRONZE).groupBy("unique_key").count().filter("count > 1")
print("keys appearing in more than one batch:", dups.count())
dups.show(5)

keys appearing in more than one batch: 0
+----------+-----+
|unique_key|count|
+----------+-----+
+----------+-----+



In [0]:
display(spark.table(LOG_TABLE).orderBy(col("start_time").desc()))

layer,parameter,start_time,end_time,status,rows_read,rows_inserted,rows_updated,error_message
Raw-to-Bronze,page_4,2026-10-08T14:43:29.223Z,2026-10-08T14:43:41.413Z,SUCCESS,50000,50000,0,null
Raw-to-Bronze,page_3,2026-10-08T14:43:09.890Z,2026-10-08T14:43:27.444Z,SUCCESS,50000,50000,0,null
Raw-to-Bronze,page_4,2026-10-08T06:58:25.365Z,2026-10-08T06:58:35.671Z,SUCCESS,50000,50000,0,null
Raw-to-Bronze,page_3,2026-10-08T06:58:12.453Z,2026-10-08T06:58:24.174Z,SUCCESS,50000,50000,0,null
Raw-to-Bronze,page_3,2026-10-08T06:55:43.018Z,2026-10-08T06:55:45.607Z,FAILURE,null,null,null,"[UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN] The feature is not supported: Queries from raw JSON/CSV/XML files are disallowed when the referenced columns only include the internal corrupt record column (named `_corrupt_record` by default). For example: `spark.read.schema(schema).json(file).filter($""_corrupt_record"".isNotNull).count()` and `spark.read.schema(schema).json(file).select(""_corrupt_record"").show()`. Instead, you can cache or save the parsed results and then send the same que"
Raw-to-Bronze,page_3,2026-10-08T06:54:35.097Z,2026-10-08T06:54:37.312Z,FAILURE,null,null,null,[NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000 JVM stacktrace: org.apache.spark.sql.AnalysisException at com.databricks.serverless.ServerlessGCEdgeCheck$.throwError(ServerlessGCEdgeCheck.scala:72) at com.databricks.serverless.ServerlessGCEdgeCheck$.checkBlockCacheCommand(ServerlessGCEdgeCheck.scala:50) at org.apache.spark.sql.connect.planner.SparkConnectPlanner.handleCreateDataFrameSnapshotCommand(SparkConnectPlanner.scala:5700) at org.
Raw-to-Bronze,page_3,2026-10-08T06:52:07.158Z,2026-10-08T06:52:09.968Z,FAILURE,null,null,null,"[UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN] The feature is not supported: Queries from raw JSON/CSV/XML files are disallowed when the referenced columns only include the internal corrupt record column (named `_corrupt_record` by default). For example: `spark.read.schema(schema).json(file).filter($""_corrupt_record"".isNotNull).count()` and `spark.read.schema(schema).json(file).select(""_corrupt_record"").show()`. Instead, you can cache or save the parsed results and then send the same que"


## 7. Reset (only if needed)

Run this if the Bronze table was created by an older version of the notebook
without partitioning, or if you want to rebuild from scratch. Everything can be
reloaded from the volume afterwards by running section 5 again.

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {BRONZE}")
# spark.sql(f"DROP TABLE IF EXISTS {QUARANTINE}")
# print("dropped -- re-run section 5 to rebuild from the volume")